# Standard Gibbs Sampling for Potts Graph Coloring

This notebook explains the **standard Gibbs sampler** used in our project.

The focus is the part that is different from Metropolis, lifting, and Chromatic sampling:
$$
\boxed{
\begin{aligned}
&\text{choose one vertex}\\
&\longrightarrow\ \text{calculate all }q\text{ color probabilities}\\
&\longrightarrow\ \text{draw one color directly}
\end{aligned}
}
$$

All formulas below were checked against Gonzalez et al. (2011) and Faizi et al. (2020). Formulas adapted to our graph-coloring model are marked as project derivations.

# 1. Model Shared by All of Our Solvers

Let
$$
G=(V,E),
\qquad
n=|V|
$$

Each vertex has one of $q$ Potts color labels:
$$
\sigma_v\in\{1,2,\ldots,q\},
\qquad q\ge3
$$

The whole coloring is
$$
\sigma=(\sigma_v)_{v\in V}
$$

The set of all possible colorings is
$$
\Omega=\{1,2,\ldots,q\}^{V}
$$

Our project gives one positive penalty $J$ to every conflicting edge:
$$
H(\sigma)
=
J\sum_{\{u,v\}\in E}
\mathbf 1\{\sigma_u=\sigma_v\},
\qquad J>0
$$

When $J=1$,
$$
H(\sigma)=\#\text{ conflicting edges}
$$

The Boltzmann distribution is
$$
\pi(\sigma)
=
\frac{e^{-\beta H(\sigma)}}{Z},
\qquad
Z=\sum_{\tau\in\Omega}e^{-\beta H(\tau)}
$$

where
$$
\beta=\frac{1}{\mathcal T}
$$

and $\mathcal T$ is the temperature.

Faizi et al. write their lattice Potts Hamiltonian as
$$
H_{\mathrm{paper}}(\sigma)
=
-\sum_{\langle k,l\rangle}
J_{kl}\,\delta(\sigma_k,\sigma_l)
$$

Here $\delta(a,b)=1$ when $a=b$, and $0$ otherwise.

For our antiferromagnetic graph-coloring model, replace the lattice neighbor pairs by graph edges and set $J_{uv}=-J$. This gives the positive conflict energy above.

This model setup is shared with the earlier Chromatic notebook. The remaining sections focus only on Gibbs sampling.

# 2. To Update One Vertex, Only Its Neighbors Matter

The neighbors of vertex $v$ are
$$
\mathcal N(v)
=
\{u\in V:\{u,v\}\in E\}
$$

All labels except the label at $v$ are written as
$$
\sigma_{-v}
$$

Gonzalez et al., Equation (2.2), gives the Markov-blanket rule:
$$
\pi(\sigma_v\mid\sigma_{-v})
=
\pi(\sigma_v\mid\sigma_{\mathcal N(v)})
$$

In simple words: to calculate a new color probability for $v$, Gibbs only needs the colors of the neighbors of $v$.

Their general Gibbs rule, Equation (2.3), is
$$
X_v
\sim
\pi\!\left(
X_v\mid X_{\mathcal N(v)}=x_{\mathcal N(v)}
\right)
\propto
\prod_{A:\,v\in A}
f_A\!\left(X_v,x_{\mathcal N(v)}\right)
$$

For our Potts model, these local factors are the edges touching $v$.

# 3. Count the Neighbors with Each Color

For a possible color $c$, define
$$
n_v(c;\sigma)
=
\sum_{u\in\mathcal N(v)}
\mathbf 1\{\sigma_u=c\}
$$

$n_v(c;\sigma)$ is the number of neighbors of $v$ that currently have color $c$.

If $v$ is changed to color $c$, its local conflict energy is
$$
H_v(c;\sigma)
=
J\,n_v(c;\sigma)
$$

So a smaller $n_v(c;\sigma)$ means that color $c$ would create fewer conflicts at $v$.

# 4. Main Gibbs Probability for Our Potts Model

Let $\sigma^{v\rightarrow c}$ mean that only vertex $v$ is changed to color $c$.

The total energy can be written as
$$
H(\sigma^{v\rightarrow c})
=
H_{-v}(\sigma_{-v})
+
J\,n_v(c;\sigma)
$$

$H_{-v}$ does not depend on the new color $c$.

Start with the Gibbs conditional:
$$
\pi(\sigma_v=c\mid\sigma_{-v})
=
\frac{
e^{-\beta H(\sigma^{v\rightarrow c})}
}{
\displaystyle\sum_{d=1}^{q}
e^{-\beta H(\sigma^{v\rightarrow d})}
}
$$

The common term containing $H_{-v}$ cancels. Therefore
$$
\boxed{
g_v(c\mid\sigma_{-v})
=
\frac{
e^{-\beta J n_v(c;\sigma)}
}{
\displaystyle\sum_{d=1}^{q}
e^{-\beta J n_v(d;\sigma)}
}
}
$$

This boxed equation is the main Gibbs function used by our project.

It is **derived for our project** by putting our Potts conflict energy into the Gibbs conditional from Gonzalez et al., Equation (2.3), and Faizi et al., Equation (7).

# 5. What the Symbols Mean

- $v$ is the vertex being updated.
- $c$ is one possible new color.
- $d$ is a temporary index that runs through all $q$ colors.
- $\sigma_{-v}$ means all colors except the color at $v$.
- $\mathcal N(v)$ is the set of neighbors of $v$.
- $n_v(c;\sigma)$ counts how many neighbors of $v$ have color $c$.
- $J$ is the conflict penalty for one edge.
- $\beta=1/\mathcal T$ is the inverse temperature.
- $g_v(c\mid\sigma_{-v})$ is the probability that Gibbs assigns color $c$ to $v$.

It is also useful to define a score before dividing by the total:
$$
w_v(c)
=
e^{-\beta Jn_v(c;\sigma)}
$$

Then
$$
g_v(c\mid\sigma_{-v})
=
\frac{w_v(c)}{\displaystyle\sum_{d=1}^{q}w_v(d)}
$$

# 6. One Random-Scan Gibbs Update

Faizi et al., Algorithm 2, chooses one **component** uniformly. In our graph model, one component means one vertex.

First choose
$$
v_t\sim\operatorname{Uniform}(V)
$$

so
$$
\Pr(v_t=v)=\frac{1}{n}
$$

Then draw the new color from all $q$ choices:
$$
\sigma_{v_t}^{(t+1)}
\sim
g_{v_t}\!\left(\,\cdot\mid\sigma_{-v_t}^{(t)}\right)
$$

Every other vertex keeps its color:
$$
\sigma_u^{(t+1)}
=
\sigma_u^{(t)},
\qquad u\ne v_t
$$

The current color is one of the $q$ choices. Therefore Gibbs can choose the same color again.

# 7. Gibbs Transition Function

For a selected vertex $v$, the transition is
$$
T_v(\sigma'\mid\sigma)
=
\mathbf 1\{\sigma'_{-v}=\sigma_{-v}\}
\,g_v(\sigma'_v\mid\sigma_{-v})
$$

The indicator means that only $v$ is allowed to change.

Because each vertex is selected with probability $1/n$, the full random-scan transition is
$$
\boxed{
T_{\mathrm{GS}}(\sigma'\mid\sigma)
=
\frac{1}{n}
\sum_{v\in V}T_v(\sigma'\mid\sigma)
}
$$

If the next state changes only $v$ to color $c$, then
$$
T_{\mathrm{GS}}(\sigma^{v\rightarrow c}\mid\sigma)
=
\frac{1}{n}
g_v(c\mid\sigma_{-v})
$$

If Gibbs draws the current color again, then $\sigma'=\sigma$. This is a valid self-transition, not a rejected move.

# 8. Why Gibbs Has Acceptance Probability 1

After vertex $v$ has already been selected, the Gibbs proposal is
$$
Q_v(c\mid\sigma)
=
g_v(c\mid\sigma_{-v})
$$

The complete random-scan proposal also has the factor $1/n$, but that factor cancels in the forward and reverse ratio.

Let the current color be $a=\sigma_v$, and let $\sigma'=\sigma^{v\rightarrow c}$. The Metropolis-Hastings ratio is
$$
R
=
\frac{
\pi(\sigma')Q_v(a\mid\sigma')
}{
\pi(\sigma)Q_v(c\mid\sigma)
}
$$

Using
$$
\pi(\sigma)
=
\pi(\sigma_{-v})g_v(a\mid\sigma_{-v})
$$

and
$$
\pi(\sigma')
=
\pi(\sigma_{-v})g_v(c\mid\sigma_{-v}),
$$

the terms cancel:
$$
R=1
$$

Therefore
$$
\boxed{
A_v(c\mid a)
=
\min(R,1)
=
1
}
$$

This is Faizi et al., Equation (6). It is true because Gibbs proposes from the exact conditional distribution. It is not true for a general uniform-color Metropolis proposal.

# 9. Detailed Balance

Suppose $\sigma$ and $\sigma'$ differ only at vertex $v$, with colors $a$ and $c$.

Then
$$
\begin{aligned}
\pi(\sigma)T_v(\sigma'\mid\sigma)
&=
\pi(\sigma_{-v})
g_v(a\mid\sigma_{-v})
g_v(c\mid\sigma_{-v})\\
&=
\pi(\sigma')T_v(\sigma\mid\sigma')
\end{aligned}
$$

So the one-vertex update satisfies
$$
T_v(\sigma'\mid\sigma)\pi(\sigma)
=
T_v(\sigma\mid\sigma')\pi(\sigma')
$$

The random-scan Gibbs transition also satisfies detailed balance:
$$
T_{\mathrm{GS}}(\sigma'\mid\sigma)\pi(\sigma)
=
T_{\mathrm{GS}}(\sigma\mid\sigma')\pi(\sigma')
$$

Therefore the Boltzmann distribution is stationary:
$$
\pi T_{\mathrm{GS}}=\pi
$$

This means that at a fixed positive temperature, Gibbs preserves the correct Boltzmann distribution.

# 10. Numerical Example

Suppose
$$
q=3,
\qquad
J=1,
\qquad
\beta=1
$$

Assume that the neighbor colors of $v$ are
$$
(1,1,2)
$$

Then
$$
n_v(1;\sigma)=2,
\qquad
n_v(2;\sigma)=1,
\qquad
n_v(3;\sigma)=0
$$

The three scores are
$$
w_v(1)=e^{-2},
\qquad
w_v(2)=e^{-1},
\qquad
w_v(3)=1
$$

The total is
$$
e^{-2}+e^{-1}+1\approx1.5032
$$

So the Gibbs probabilities are
$$
\boxed{
\begin{aligned}
g_v(1\mid\sigma_{-v})&\approx0.0900,\\
g_v(2\mid\sigma_{-v})&\approx0.2447,\\
g_v(3\mid\sigma_{-v})&\approx0.6652.
\end{aligned}
}
$$

Color $3$ is most likely because it creates no local conflict. It is not certain because the temperature is still positive.

# 11. Gibbs Compared with Metropolis

Let the current color be $a=\sigma_v$.

If our Metropolis baseline proposes uniformly among the $q-1$ different colors, then
$$
Q_{\mathrm M}(b\mid a)
=
\frac{1}{q-1},
\qquad b\ne a
$$

Its local energy change is
$$
\Delta H_v(a\rightarrow b)
=
J\left[n_v(b;\sigma)-n_v(a;\sigma)\right]
$$

and its acceptance probability is
$$
A_{\mathrm M}(b\mid a)
=
\min\left(1,e^{-\beta\Delta H_v(a\rightarrow b)}\right)
$$

Gibbs instead calculates the probabilities of all $q$ colors:
$$
g_v(c\mid\sigma_{-v})
=
\frac{e^{-\beta Jn_v(c;\sigma)}}
{\displaystyle\sum_{d=1}^{q}e^{-\beta Jn_v(d;\sigma)}}
$$

and draws one color directly.

Therefore
$$
\boxed{
\begin{aligned}
\text{Metropolis: }&\text{propose one color, then accept or reject},\\
\text{Gibbs: }&\text{weigh all }q\text{ colors, then draw directly}.
\end{aligned}
}
$$

# 12. What the Other Methods Change

## Metropolized Gibbs

Faizi et al. also study **Metropolized Gibbs**. It removes the current color from the proposal and then adds a Metropolis-Hastings correction.

That is a different algorithm from the standard Gibbs baseline in this notebook.

## Lifting

Standard Gibbs uses the state
$$
\sigma
$$

Faizi's two-replica lifting construction adds an auxiliary variable:
$$
(\sigma,\epsilon),
\qquad
\epsilon\in\{+1,-1\}
$$

with extended target
$$
\widetilde\pi(\sigma,\epsilon)
=
\frac{1}{2}\pi(\sigma)
$$

The two replicas can be related by skewed detailed balance:
$$
\pi(\sigma)T_{\epsilon}(\sigma'\mid\sigma)
=
\pi(\sigma')T_{-\epsilon}(\sigma\mid\sigma')
$$

The complete lifted transition needs additional rules. Those rules belong in the lifting notebook, not in this standard Gibbs baseline.
